<div style="width:100%; text-align:center; padding:10px 0;">
<img src="project_header.png" style="width:100%; max-width:100vw; height:auto; display:block; margin:0 auto;">
</div>

# EO Africa SWAM project 
## Read MDN Chla, reproject to WGS84, clip to shapefile

#### This notebook does the following:

* Read the Chl-a output from *Chl_Step3_MDN.ipynb*
* Reproject the Chl-a image, i.e., convert curvilinear to grid data, to correct the geolocation
* clip the image according to the shapefile *EO_Africa_lake_polygon_updated.geojson*
* save output CHL as netcdf file

#### Requirements: 
* Make sure that you have already created some "*_chl.nc" output files using the *Chl_Step3_MDN.ipynb* notebook.
* the file *EO_Africa_lake_polygon_updated.geojson* is used to clip to the region of interest

#### Settings to adjust manually:
* In code cell 3 define the lake/dam name  (options: ZV, RV, TW, MV, VV, CW), default is TW. 
* Input and output paths are selected and created automatically
* Shapefile should be selected authomatically based on defined lake name

##### Authors:
**Dalin Jiang**, University of Stirling, UK;
updated for Chl-a processing by **Marie Smith**, CSIR, South Africa

In [ ]:
import os
import glob
import time
from datetime import datetime
import xarray as xr
import numpy as np
import numba
from tqdm.auto import tqdm
from dask.distributed import Client
from scipy.interpolate import griddata
from scipy.spatial import cKDTree
import geopandas as gpd
import rioxarray as rxr
import re


In [ ]:
client = Client()
client

Define the lake/dam name, the input and output paths, and the shapefile:

In [ ]:
# input/output path
lake_name = "TW"

## create the folder where the outputs will be saved:
if not os.path.exists(lake_name+'_chl'):
    os.makedirs(lake_name+'_chl')
    print(f"Created folder: {lake_name}_chl")
else:
    print(f"Folder {lake_name}_chl already exists.") [web:1][web:5]

in_path = "/"+lake_name+"_mdn/"      
out_path = "/"+lake_name+"_chl/"

# lake shape file to crop the image
roi_file = "EO_Africa_lake_polygon_updated.geojson"
roi = gpd.read_file(roi_file)

Define functions for converting to curvilinear grid and clipping to shapefile:

In [ ]:
@numba.njit

def convert_curvilinear_to_grid(ds, var_name, max_dist=0.001):
    """
    Regrids data and masks pixels too far from actual measurements.
    max_dist: Distance threshold in coordinate units (e.g., 0.001 deg ≈ 100m).
    """
    chl = ds[var_name].load()
    
    # Target grid setup
    ysize, xsize = chl.shape
    lon1d = np.linspace(ds.lon.min().item(), ds.lon.max().item(), xsize)
    lat1d = np.linspace(ds.lat.min().item(), ds.lat.max().item(), ysize)
    lon_grid, lat_grid = np.meshgrid(lon1d, lat1d)
    
    # Source points
    lon_2d, lat_2d = np.meshgrid(ds.lon.values, ds.lat.values)
    source_values = chl.values.ravel()
    valid = ~np.isnan(source_values)
    
    points = np.column_stack([lon_2d.ravel()[valid], lat_2d.ravel()[valid]])
    values = source_values[valid]

    # SOLUTION 1: Prevent QhullError by checking point count
    if len(points) < 4:
        regridded = np.full((ysize, xsize), np.nan)
    else:
        # Interpolate
        regridded = griddata(points, values, (lon_grid, lat_grid), method='linear')
        
        # SOLUTION 2: Mask "False Data" using a distance threshold
        tree = cKDTree(points)
        target_coords = np.column_stack([lon_grid.ravel(), lat_grid.ravel()])
        dist, _ = tree.query(target_coords)
        dist_mask = dist.reshape(lon_grid.shape)
        
        # Nullify pixels that are too far from any real measurement
        regridded[dist_mask > max_dist] = np.nan
    
    return xr.Dataset(
        {var_name: (('y', 'x'), regridded)},
        coords={'x': lon1d, 'y': lat1d}
    )
 
def crop_lake_and_add_info(ds, lake_shape, lake_name):
    """
    ds: the raw image to be cropped
    lake_shape: the shape file containing all study lakes
    lake_name: short name of the lake, to get a single lake shape from the shape file, i.e., ZV, TW, RV
    """
    ds = ds.rio.write_crs("EPSG: 4326")
    lake_roi = lake_shape[lake_shape["Short_name"] == lake_name]
    
    try:
        mk_arr = ds["CHL"].rio.clip(lake_roi.geometry.values, drop=True)
        if mk_arr.size == 0:
            print(f"Empty clip for {lake_name}")
            return None
    except Exception as e:
        print(f"Clip failed for {lake_name}: {e}")
        return None
    
    mk_dt = mk_arr.to_dataset(name="CHL")

    # convert data to float32 to save space
    for var in ['x','y','CHL']:
        mk_dt[var] = mk_dt[var].astype('float32')

    # change coordinates name, otherwise SNAP won't read correctly
    mk_dt = mk_dt.rename({'y':'lat', 'x':'lon'})
    
    # add attributes
    mk_dt["CHL"].attrs={'short_name':'CHL',
                       'standard_name':'Chlorophyll a concentration',
                       'units': 'mgm-3',
                       'grid_mapping':'spatial_ref'}

    # add global attributes
    mk_dt.attrs = {'Product': 'Chlorophyll a concentration',
                   'Sensor': 'Sentinel2-MSI',
                   'Project': 'ESA EO Africa R&D SWAM'
                   }

    return(mk_dt)

In [ ]:
all_img = glob.glob(in_path+"*.nc",recursive=True)
print(f' found {len(all_img)} images, with the first 5 showing below:')
all_img[0:5]

Below is the main step for batch processing all files, including reading the input file, regridding the data, clipping the data, and write the netcdf output files. This can take a few minutes to complete because it is running over all the files, so be patient! 


In [ ]:
# process all images
for one_img in tqdm(all_img):
    start_time = time.time()
    
    #one_dt = xr.open_dataset(one_img)        #,chunks={"x": 512, "y": 512}
    one_dt = xr.open_dataset(one_img, chunks={'time': 1, 'y': 1000, 'x': 1000})

    # get image information: name of lake, date and time
    img_info = os.path.basename(one_img)
    parts = img_info.split('_')
    mission = parts[0]      # S2A
    year, month, day = parts[2:5]    # 2021, 03, 31
    hour, minute, second = parts[5:8]  # 08, 49, 50
    tile = parts[8]     # T34HBH
    time_stamp = datetime.strptime(''.join(img_info.split('_')[2:8]), "%Y%m%d%H%M%S") 
    new_name = f"{mission}_{year}{month}{day}T{hour}{minute}{second}_{tile}_ACO_{lake_name}_CHL.nc"
    
    ## Access flags and chlorophyll as Dask arrays (do NOT call .values)
    chl = one_dt['chl']


    # Create mask for chlorophyll > 0 (lazy operation)
    data_mask = chl > 0
    if data_mask.sum().load() == 0:
        print(f"Skipping {img_info} - no CHL > 0")
        one_dt.close()
        continue
        

    # Use combined_mask to mask chlorophyll (this remains lazy)
    masked_chla = chl.where(data_mask)
    raw_dt = masked_chla.to_dataset(name='CHL')

    # convert the dat to a regular grid 
    reg_dt = convert_curvilinear_to_grid(ds=raw_dt, var_name='CHL')
    water_dt = crop_lake_and_add_info(ds=reg_dt, lake_shape=roi, lake_name=lake_name)
    if water_dt is None:
        print(f"Skipping {img_info} - no lake data")
        continue
    
    water_dt = water_dt.expand_dims(dim={'time':[time_stamp]})
    
    comp = dict(zlib=True, complevel=6, dtype='float32')  # 1–9 (higher = more compression, slower)
    encoding = {var: comp for var in water_dt.data_vars}

    out_file = out_path + new_name
    water_dt.to_netcdf(out_file,format='NETCDF4', encoding=encoding)

    end_time = time.time()
    print(f"processing {img_info } used %s" % (end_time - start_time))
